In [1]:
import pandas as pd

train = pd.read_pickle("../data/train.pkl")
test = pd.read_pickle("../data/test.pkl")

train.head(), test.head()


(       user   item  rating
 38240  1440  39757       1
 3806   5297  42139       4
 27927  2712  78915       5
 6006   9949  77259       5
 65809  5741  72446       5,
        user   item  rating
 33967   454  23650       5
 22853  1329   5719       4
 19448  2406  35799       2
 9732   9563  25908       5
 7129   5171  57947       5)

In [2]:
from sklearn.metrics import mean_squared_error
import numpy as np

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


In [3]:
global_mean = train["rating"].mean()
global_mean


4.343230032627855

In [4]:
y_true = test["rating"].values
y_pred = [global_mean] * len(test)

rmse_global = rmse(y_true, y_pred)
rmse_global


0.9939134286799708

In [5]:
user_mean = train.groupby("user")["rating"].mean()


In [6]:
def predict_user_mean(row):
    u = row["user"]
    return user_mean.get(u, global_mean)


In [7]:
y_pred = test.apply(predict_user_mean, axis=1)
rmse_user = rmse(test["rating"], y_pred)
rmse_user


0.9362146727500442

In [8]:
item_mean = train.groupby("item")["rating"].mean()


In [9]:
def predict_item_mean(row):
    i = row["item"]
    return item_mean.get(i, global_mean)


In [10]:
y_pred = test.apply(predict_item_mean, axis=1)
rmse_item = rmse(test["rating"], y_pred)
rmse_item


1.052213984020488

In [11]:
!pip install implicit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.4/761.4 kB 17.5 MB/s  0:00:00


In [12]:
import scipy.sparse as sp
from implicit.als import AlternatingLeastSquares

# matrix usuario × item
train_matrix = sp.coo_matrix(
    (train["rating"].values,
     (train["user"].values, train["item"].values))
)

model = AlternatingLeastSquares(
    factors=50,
    regularization=0.1,
    iterations=20,
    random_state=42
)

model.fit(train_matrix)


/Users/andresrivadeneyra/miniconda3/envs/recsys/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/andresrivadeneyra/miniconda3/envs/recsys/lib/python3.10/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
/Users/andresrivadeneyra/miniconda3/envs/recsys/lib/python3.10/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.006196260452270508 seconds
  warnings.warn(
100%|██████████| 20/20 [00:03<00

In [13]:
def predict_als(row):
    u = row["user"]
    i = row["item"]
    return model.user_factors[u] @ model.item_factors[i]


In [15]:
y_pred = test.apply(predict_als, axis=1)
rmse_als = rmse(test["rating"], y_pred)
rmse_als


4.4353064904864

In [16]:
results = {
    "Global Mean": rmse_global,
    "User Mean": rmse_user,
    "Item Mean": rmse_item,
    "ALS": rmse_als
}

results


{'Global Mean': 0.9939134286799708,
 'User Mean': 0.9362146727500442,
 'Item Mean': 1.052213984020488,
 'ALS': 4.4353064904864}

In [17]:
def recommend_for_user(user_id, n=10):
    user_items = train[train["user"] == user_id]["item"].unique()
    all_items = set(train["item"].unique())

    # items no vistos
    items_to_score = list(all_items - set(user_items))

    scores = [
        (item, model.user_factors[user_id] @ model.item_factors[item])
        for item in items_to_score
    ]

    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    return scores[:n]

recommend_for_user(10)


[(61047, 0.008245705),
 (59764, 0.0056852014),
 (61053, 0.0055134543),
 (61447, 0.00482815),
 (4177, 0.004798343),
 (41806, 0.004796014),
 (37287, 0.004233858),
 (58774, 0.0039970204),
 (46126, 0.003884867),
 (51384, 0.0038717454)]